In [ ]:
# GPU'yu kontrol et
!nvidia-smi

# Unsloth ve bağımlılıkların güncel/hızlı kurulumu
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

Thu Apr  2 09:44:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             69W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Veri setindeki cümle uzunluğuna göre artırılabilir
dtype = None # Otomatik algılama
load_in_4bit = True # VRAM tasarrufu için 4-bit yükleme

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.1-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# LoRA (Low-Rank Adaptation) parametrelerini ekliyoruz
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank (16, 32 veya 64 denenebilir)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("json", data_files={
    "train": "mizan_6class_train.jsonl",
    "validation": "mizan_6class_validation.jsonl"
})

dataset = dataset.map(formatting_prompts_func, batched = True,)

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

print(f"✅ Veri setleri hazır!")
print(f"📊 Train: {len(train_dataset)} satır")
print(f"📊 Validation: {len(eval_dataset)} satır")

✅ Veri setleri hazır!
📊 Train: 7006 satır
📊 Validation: 1501 satır


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,

        num_train_epochs = 1,

        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,

        eval_strategy = "steps",
        eval_steps = 50,

        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Motor ateşlendi, 1 TAM EPOCH'luk asıl eğitim başlıyor...")
trainer_stats = trainer.train()

🚀 Motor ateşlendi, 1 TAM EPOCH'luk asıl eğitim başlıyor...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,006 | Num Epochs = 1 | Total steps = 438
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss
50,0.256882,0.271230
100,0.272336,0.266175
150,0.265602,0.263612
200,0.260067,0.261802
250,0.251686,0.260825
300,0.247887,0.259587
350,0.254763,0.258980
400,0.258327,0.258467


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

In [ ]:
from google.colab import drive

# Google Drive'ı Colab'e monte ediyoruz
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil
import glob

# Drive içinde güvenli limanımızı (klasörü) oluşturuyoruz
drive_path = "/content/drive/MyDrive/Mizan_V3_Step10_Model"
os.makedirs(drive_path, exist_ok=True)

print("⏳ 1. Model LoRA formatında kaydediliyor...")
# LoRA olarak Colab içine kaydet
model.save_pretrained("mizan_v3_step10_lora")
tokenizer.save_pretrained("mizan_v3_step10_lora")

# LoRA klasörünü Drive'a kopyala
shutil.copytree("mizan_v3_step10_lora", f"{drive_path}/mizan_v3_step10_lora", dirs_exist_ok=True)
print("✅ LoRA formatı başarıyla Drive'a yedeklendi!\n")

print("⏳ 2. Model GGUF (4-bit) formatında kaydediliyor... (Bu işlem 5-10 dakika sürebilir)")
# GGUF olarak Colab içine kaydet (Q4_K_M en optimum boyuttur)
model.save_pretrained_gguf("mizan_v3_step10", tokenizer, quantization_method = "q4_k_m")

# Oluşan .gguf uzantılı dosyayı bul ve Drive'a kopyala
gguf_files = glob.glob("*.gguf")
for gguf_file in gguf_files:
    shutil.copy(gguf_file, drive_path)
    print(f"✅ {gguf_file} başarıyla Drive'a yedeklendi!")

print(f"\n🎉 TEBRİKLER! Tüm modeller Google Drive'ına ({drive_path}) güvenle kaydedildi.")

⏳ 1. Model LoRA formatında kaydediliyor...
✅ LoRA formatı başarıyla Drive'a yedeklendi!

⏳ 2. Model GGUF (4-bit) formatında kaydediliyor... (Bu işlem 5-10 dakika sürebilir)
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 20738.22it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:07<00:00, 16.88s/it]


Unsloth: Merge process complete. Saved to `/content/mizan_v3_step10`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['mizan_v3_step10_gguf/Llama-3.1-8B.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['mizan_v3_step10_gguf/Llama-3.1-8B.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/Llama-3.1-8B'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model mizan_v3_step10_gguf/Llama-3.1-8B.Q4_K_M.gguf -p "why is the sky blue?"

🎉 TEBRİKLER! Tüm modeller Google Drive'ına (/content/drive/MyDrive/Mizan_V3_Step10_Model) güvenle kaydedildi.


In [ ]:
from google.colab import files

# Drive'a attığımız GGUF dosyasını bilgisayara da indiriyoruz
gguf_files = glob.glob("*.gguf")
for gguf_file in gguf_files:
    print(f"⬇️ {gguf_file} bilgisayarınıza indiriliyor...")
    files.download(gguf_file)

In [ ]:
import os
import shutil

# Drive yolunu tekrar tanımlayalım
drive_path = "/content/drive/MyDrive/Mizan_V3_Step10_Model"

print("🔍 GGUF dosyası aranıyor...")
# Unsloth'un oluşturduğu klasördeki .gguf dosyasını bulalım
gguf_kaynak = "mizan_v3_step10_gguf/Llama-3.1-8B.Q4_K_M.gguf"

if os.path.exists(gguf_kaynak):
    print("✅ Dosya bulundu, Drive'a kopyalanıyor...")
    # Hedef dosya adını net bir şekilde belirliyoruz
    hedef_yol = os.path.join(drive_path, "Mizan_V3_Final_Model.gguf")
    shutil.copy(gguf_kaynak, hedef_yol)
    print(f"🚀 İŞLEM TAMAM! Drive'da şu isimle bak: Mizan_V3_Final_Model.gguf")
else:
    print("❌ Kaynak dosya bulunamadı! Lütfen sol taraftaki dosya simgesine (klasör) tıklayıp 'mizan_v3_step10_gguf' klasörünün içine bak.")

🔍 GGUF dosyası aranıyor...
✅ Dosya bulundu, Drive'a kopyalanıyor...
🚀 İŞLEM TAMAM! Drive'da şu isimle bak: Mizan_V3_Final_Model.gguf


In [ ]:
# Modeli çıkarım (inference) yani tahmin moduna alıyoruz
FastLanguageModel.for_inference(model)

# İngilizce test cümlemiz (İçinde bilerek typographic ve lexical hatalar bıraktık)
test_cumlesi = "The nlp enginer is anlyzing the chat datas."

# Alpaca formatına sokuyoruz
prompt = alpaca_prompt.format(
    "You are an expert text correction system. Correct the input text and provide a detailed JSON analysis.", # Instruction
    test_cumlesi, # Input
    "" # Response (Model burayı dolduracak)
)

print("⏳ Model düşünüyor...\n")

# Modeli çalıştır
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
prediction = tokenizer.batch_decode(outputs)

# Sadece modelin ürettiği cevabı (JSON kısmını) ekrana yazdır
final_output = prediction[0].split("### Response:")[1].replace(EOS_TOKEN, "").strip()

print("🎯 İŞTE MODELİN İNGİLİZCE ÇIKTISI:\n")
print(final_output)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⏳ Model düşünüyor...

🎯 İŞTE MODELİN İNGİLİZCE ÇIKTISI:

{"originalText": "The nlp enginer is anlyzing the chat datas.", "correctedText": "The NLP engineer is analyzing the chat data.", "corrections": [{"original": "The nlp enginer is anlyzing the chat datas.", "corrected": "The NLP engineer is analyzing the chat data.", "type": "typographic", "explanation": "Fixed typographic error."}]}
